# Nhiệm vụ N7: Chạy Lưới Đánh Giá Semi-supervised K-Means cho Bản C, D, A+C, A+D
### Đồ án Tổng hợp: Generalized Category Discovery (GCD) - TextGCD Leak Analysis

**Tác giả thực hiện:** Kiệt (Nhiệm vụ N7)

**Mục tiêu thí nghiệm:**
Đánh giá hiệu năng phân cụm Semi-supervised K-Means khi sử dụng mô tả text từ Lexicon đã lọc rò rỉ tên lớp:
- **Bản C:** Lexicon đã bỏ tag lớp New và cắt tiền tố tên khỏi thuộc tính lớp New (do Giang tạo ở N3).
- **Bản D:** Mọi thuộc tính đã cắt tiền tố, không còn bất kỳ tag nào (bản sạch hoàn toàn).
Thí nghiệm thực hiện trên 3 tập dữ liệu:
1. **CUB-200-2011** (`cub` - 200 lớp)
2. **Stanford Cars** (`scars` - 196 lớp)
3. **FGVC-Aircraft** (`aircraft` - 100 lớp)

**Các bản đánh giá trong N7:**
- **Bản C (Chỉ text bản C):** `feats/{ds}_txt_C.npy`
- **Bản D (Chỉ text bản D):** `feats/{ds}_txt_D.npy`
- **Bản A+C (Bản chính, $\alpha = 0.5$):** Ghép ảnh A + text C: $F = [\sqrt{1-\alpha} F_I, \sqrt{\alpha} F_T]$
- **Bản A+D (Bản chính, $\alpha = 0.5$):** Ghép ảnh A + text D: $F = [\sqrt{1-\alpha} F_I, \sqrt{\alpha} F_T]$
- **Độ nhạy $\alpha \in \{0.25, 0.75\}$ cho A+C:** `A+C_a0.25`, `A+C_a0.75`
- **Độ nhạy $\alpha \in \{0.25, 0.75\}$ cho A+D:** `A+D_a0.25`, `A+D_a0.75`

**Quy ước thực nghiệm:**
- Split SSB cố định (`splits/{ds}.csv`) với seed 0 (50% nhãn).
- 5 seeds khởi tạo k-means: `0, 1, 2, 3, 4`.
- Chấm điểm theo SimGCD ACC `v2` (sample-weighted) trên các hàng không có nhãn (`is_labeled=0`), báo cáo `All`, `Old`, `New`.
- Tự động bỏ qua (`skip`) các thí nghiệm đã có file kết quả để chống timeout và cho phép chạy ngắt quãng/chạy từng tập.

In [ ]:
## 1. Clone Repo & Cài đặt Môi trường

# Lưu ý: Bật GPU (T4 x2 hoặc P100) và BẬT INTERNET trong Settings Notebook của Kaggle!
REPO_URL = "https://github.com/tanq1965/gcd-text-leak.git"
BRANCH = "feat/n7-eval-cd"
REPO = "/kaggle/working/gcd-text-leak"

import os, sys
if not os.path.exists(REPO):
    !git clone -b {BRANCH} {REPO_URL} {REPO}
%cd {REPO}
!git checkout {BRANCH}
!git branch --show-current

# Cài đặt các thư viện cần thiết
!pip install -q open_clip_torch pillow tqdm pandas scipy scikit-learn
!python -c "import torch, open_clip; print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available(), '| Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"

In [ ]:
## 2. Chuẩn bị Thư mục & Dữ liệu Đầu vào (feats_n4.zip & 6 file rtg/{ds}_C.pkl, rtg/{ds}_D.pkl)

import os, shutil, glob
from pathlib import Path

# Tạo các thư mục làm việc theo quy ước nhóm
os.makedirs("feats", exist_ok=True)
os.makedirs("rtg", exist_ok=True)
os.makedirs("results", exist_ok=True)

# 2.1 Giải nén hoặc sao chép đặc trưng ảnh (feats_n4.zip từ N4)
# Link Google Drive lưu trữ feats_n4.zip (trong README.md):
# https://drive.google.com/drive/folders/1iZhZTkanZov6NvgCODRuZwbILuW1HAdr?usp=sharing
zip_candidates = glob.glob("/kaggle/input/**/feats_n4*.zip", recursive=True) + glob.glob("/kaggle/working/**/feats_n4*.zip", recursive=True)
npy_inputs = glob.glob("/kaggle/input/**/*_img*.npy", recursive=True)

if zip_candidates:
    print(f"📦 Tìm thấy file zip đặc trưng ảnh: {zip_candidates[0]}")
    !unzip -q -o {zip_candidates[0]} -d feats/
elif npy_inputs:
    print(f"📦 Sao chép đặc trưng ảnh từ Kaggle Input ({len(npy_inputs)} files)...")
    for f in npy_inputs:
        shutil.copy(f, f"feats/{os.path.basename(f)}")
else:
    # Hỗ trợ tải trực tiếp qua gdown từ Google Drive nếu chưa có sẵn
    print("🌐 Đang thử tải feats_n4.zip từ Google Drive nhóm qua gdown...")
    !pip install -q gdown
    !gdown --folder "https://drive.google.com/drive/folders/1iZhZTkanZov6NvgCODRuZwbILuW1HAdr?usp=sharing" -O /kaggle/working/gdrive_feats/
    gdrive_zips = glob.glob("/kaggle/working/gdrive_feats/**/feats_n4*.zip", recursive=True)
    if gdrive_zips:
        !unzip -q -o {gdrive_zips[0]} -d feats/
    else:
        for f in glob.glob("/kaggle/working/gdrive_feats/**/*.npy", recursive=True):
            shutil.copy(f, f"feats/{os.path.basename(f)}")

# 2.2 Tìm nạp 6 file RTG bản C và D ({ds}_C.pkl, {ds}_D.pkl)
rtg_inputs = glob.glob("/kaggle/input/**/*_[CD].pkl", recursive=True) + glob.glob("/kaggle/working/**/*_[CD].pkl", recursive=True)
for f in rtg_inputs:
    dest = Path("rtg") / os.path.basename(f)
    if not dest.exists():
        shutil.copy(f, dest)
        print(f"✔ Sao chép {f} -> {dest}")

print("\n--- THƯ MỤC LÀM VIỆC HIỆN TẠI ---")
print("Thư mục feats/:", os.listdir("feats"))
print("Thư mục rtg/:  ", os.listdir("rtg"))

In [ ]:
## 2.3 Sanity Check Dữ liệu Đầu vào

import numpy as np, pandas as pd

DATASETS_CHECK = ["cub", "scars", "aircraft"]
EXPECTED_COUNTS = {"cub": 5994, "scars": 8144, "aircraft": 6667}

print("=== KIỂM TRA ĐẶC TRƯNG ẢNH & 6 FILE RTG C/D ===")
for ds in DATASETS_CHECK:
    img_feat_p = f"feats/{ds}_img.npy"
    img_ids_p = f"feats/{ds}_img_ids.npy"
    split_p = f"splits/{ds}.csv"
    assert os.path.exists(split_p), f"Lỗi: Thiếu split file {split_p}!"
    
    if os.path.exists(img_feat_p) and os.path.exists(img_ids_p):
        feats = np.load(img_feat_p)
        ids = np.load(img_ids_p, allow_pickle=True)
        exp_n = EXPECTED_COUNTS[ds]
        assert feats.shape == (exp_n, 512), f"{ds}: Shape sai {feats.shape}, kỳ vọng ({exp_n}, 512)"
        norms = np.linalg.norm(feats, axis=1)
        assert np.allclose(norms, 1.0, atol=1e-3), f"{ds}: Vector ảnh chưa chuẩn hoá L2!"
        print(f"✔ [ẢNH] {ds.upper():8s}: Shape {feats.shape} | L2 norm OK | IDs: {len(ids)}")
    else:
        print(f"⚠️ [ẢNH] {ds.upper():8s}: Chưa có đặc trưng ảnh {img_feat_p}. Vui lòng giải nén feats_n4.zip!")
        
    for ver in ["C", "D"]:
        rtg_p = f"rtg/{ds}_{ver}.pkl"
        if os.path.exists(rtg_p):
            print(f"✔ [RTG {ver}] {ds.upper():8s}: File {rtg_p} sẵn sàng ({os.path.getsize(rtg_p)/1024:.1f} KB)")
        else:
            print(f"⚠️ [RTG {ver}] {ds.upper():8s}: Chưa có {rtg_p}. Hãy copy {ds}_{ver}.pkl vào thư mục rtg/!")

In [ ]:
## 3. Mã hoá Text Bản C và D (src.encode_text)

# Chọn danh sách tập muốn chạy (chạy cả 3 tập hoặc từng tập nếu cần)
DATASETS = ["cub", "scars", "aircraft"]
VERSIONS = ["C", "D"]

print("=== MÃ HOÁ TEXT BẢN C VÀ D (PROMPT TEXTGCD + CLIP ViT-B/16) ===")
for ds in DATASETS:
    split_file = f"splits/{ds}.csv"
    for ver in VERSIONS:
        out_feat = f"feats/{ds}_txt_{ver}.npy"
        out_ids = f"feats/{ds}_txt_{ver}_ids.npy"
        rtg_file = f"rtg/{ds}_{ver}.pkl"
        
        # Kiểm tra chống chạy lại / tiết kiệm thời gian
        if os.path.exists(out_feat) and os.path.exists(out_ids):
            print(f"⏩ [SKIP] Đã tồn tại đặc trưng text {ver}: {out_feat}")
            continue
            
        if not os.path.exists(rtg_file):
            print(f"⚠️ Bỏ qua {ds} bản {ver}: Không tìm thấy file {rtg_file}")
            continue
            
        print(f"\n🚀 Đang mã hoá text bản {ver} cho {ds.upper()}...")
        !python -m src.encode_text \
            --dataset {ds} \
            --split_csv {split_file} \
            --rtg_pkl {rtg_file} \
            --output_npy {out_feat} \
            --batch_size 128

# Sanity check ma trận đặc trưng text C và D
print("\n--- KIỂM TRA CHẤT LƯỢNG ĐẶC TRƯNG TEXT C & D ---")
for ds in DATASETS:
    split_df = pd.read_csv(f"splits/{ds}.csv")
    for ver in VERSIONS:
        out_feat = f"feats/{ds}_txt_{ver}.npy"
        out_ids = f"feats/{ds}_txt_{ver}_ids.npy"
        if os.path.exists(out_feat) and os.path.exists(out_ids):
            txt_feats = np.load(out_feat)
            txt_ids = np.load(out_ids, allow_pickle=True)
            assert txt_feats.shape == (len(split_df), 512), f"{ds} {ver}: Shape text sai {txt_feats.shape}!"
            assert np.allclose(np.linalg.norm(txt_feats, axis=1), 1.0, atol=1e-3), f"{ds} {ver}: Text chưa chuẩn hoá L2!"
            assert np.array_equal(txt_ids.astype(str), split_df["image_id"].astype(str).values), f"{ds} {ver}: IDs không khớp split!"
            print(f"✔ [TEXT {ver}] {ds.upper():8s}: Shape {txt_feats.shape} | L2 norm OK | Khớp 100% split IDs")

In [ ]:
## 4. Chạy Lưới Đánh Giá N7 (src.eval) - 5 Seeds & Các Biến Thể C, D, A+C, A+D

import subprocess, time, torch
from pathlib import Path

# Cấu hình danh sách tập dữ liệu muốn chạy:
# Có thể đổi thành DATASETS = ["cub"] để chạy pilot nhanh hoặc chạy từng phần
DATASETS = ["cub", "scars", "aircraft"]
SEEDS = [0, 1, 2, 3, 4]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Định nghĩa 8 cấu hình đánh giá N7:
# 1. Bản C: Chỉ text bản C
# 2. Bản D: Chỉ text bản D
# 3. Bản A+C: Ảnh + Text C (alpha = 0.5)
# 4. Bản A+D: Ảnh + Text D (alpha = 0.5)
# 5. Bản A+C_a0.25: Độ nhạy alpha = 0.25 cho A+C
# 6. Bản A+C_a0.75: Độ nhạy alpha = 0.75 cho A+C
# 7. Bản A+D_a0.25: Độ nhạy alpha = 0.25 cho A+D
# 8. Bản A+D_a0.75: Độ nhạy alpha = 0.75 cho A+D
VARIANTS = [
    {
        "name": "C",
        "desc": "Chỉ text bản C (Text C only)",
        "feats": lambda ds: f"feats/{ds}_txt_C.npy",
        "feats2": None,
        "alpha": None,
    },
    {
        "name": "D",
        "desc": "Chỉ text bản D (Text D only)",
        "feats": lambda ds: f"feats/{ds}_txt_D.npy",
        "feats2": None,
        "alpha": None,
    },
    {
        "name": "A+C",
        "desc": "Ảnh + Text bản C (alpha = 0.5)",
        "feats": lambda ds: f"feats/{ds}_img.npy",
        "feats2": lambda ds: f"feats/{ds}_txt_C.npy",
        "alpha": 0.5,
    },
    {
        "name": "A+D",
        "desc": "Ảnh + Text bản D (alpha = 0.5)",
        "feats": lambda ds: f"feats/{ds}_img.npy",
        "feats2": lambda ds: f"feats/{ds}_txt_D.npy",
        "alpha": 0.5,
    },
    {
        "name": "A+C_a0.25",
        "desc": "Độ nhạy alpha = 0.25 cho A+C",
        "feats": lambda ds: f"feats/{ds}_img.npy",
        "feats2": lambda ds: f"feats/{ds}_txt_C.npy",
        "alpha": 0.25,
    },
    {
        "name": "A+C_a0.75",
        "desc": "Độ nhạy alpha = 0.75 cho A+C",
        "feats": lambda ds: f"feats/{ds}_img.npy",
        "feats2": lambda ds: f"feats/{ds}_txt_C.npy",
        "alpha": 0.75,
    },
    {
        "name": "A+D_a0.25",
        "desc": "Độ nhạy alpha = 0.25 cho A+D",
        "feats": lambda ds: f"feats/{ds}_img.npy",
        "feats2": lambda ds: f"feats/{ds}_txt_D.npy",
        "alpha": 0.25,
    },
    {
        "name": "A+D_a0.75",
        "desc": "Độ nhạy alpha = 0.75 cho A+D",
        "feats": lambda ds: f"feats/{ds}_img.npy",
        "feats2": lambda ds: f"feats/{ds}_txt_D.npy",
        "alpha": 0.75,
    },
]

total_configs = len(DATASETS) * len(VARIANTS) * len(SEEDS)
print(f"=== BẮT ĐẦU CHẠY LƯỚI ĐÁNH GIÁ N7 ({total_configs} LẦN CHẠY TRÊN {DEVICE.upper()}) ===")
count = 0
skipped = 0
start_time = time.time()

for ds in DATASETS:
    f_img = f"feats/{ds}_img.npy"
    if not os.path.exists(f_img):
        print(f"❌ Thiếu {f_img}! Bỏ qua tập {ds}")
        continue

    print(f"\n=========================================")
    print(f"📂 TẬP DỮ LIỆU: {ds.upper()}")
    print(f"=========================================")

    for v in VARIANTS:
        v_name = v["name"]
        feats1 = v["feats"](ds)
        feats2 = v["feats2"](ds) if v["feats2"] else None
        alpha = v["alpha"]

        if not os.path.exists(feats1):
            print(f"⚠️ Thiếu file {feats1}, bỏ qua biến thể {v_name} tập {ds}!")
            continue
        if feats2 and not os.path.exists(feats2):
            print(f"⚠️ Thiếu file {feats2}, bỏ qua biến thể {v_name} tập {ds}!")
            continue

        for seed in SEEDS:
            count += 1
            out_json = f"results/{ds}_{v_name}_s{seed}.json"

            # TÍNH NĂNG CHỐNG TIMEOUT: Bỏ qua nếu đã có kết quả
            if os.path.exists(out_json):
                skipped += 1
                print(f"⏩ [{count}/{total_configs}] [SKIP] {out_json} đã tồn tại.")
                continue

            cmd = [
                sys.executable, "-m", "src.eval",
                "--dataset", ds,
                "--variant", v_name,
                "--feats", feats1,
                "--seed", str(seed),
                "--device", DEVICE,
            ]
            if feats2:
                cmd.extend(["--feats2", feats2, "--alpha", str(alpha)])

            cmd_str = " ".join(cmd)
            t0 = time.time()
            res = subprocess.run(cmd, capture_output=True, text=True)
            elapsed = time.time() - t0

            if res.returncode == 0:
                out_lines = [l for l in res.stdout.strip().splitlines() if "All" in l]
                summary_line = out_lines[-1] if out_lines else f"Xong trong {elapsed:.1f}s"
                print(f"✔ [{count}/{total_configs}] {summary_line} ({elapsed:.1f}s)")
            else:
                print(f"❌ [{count}/{total_configs}] LỖI khi chạy: {cmd_str}")
                print("Stderr:", res.stderr[-500:])

total_elapsed = time.time() - start_time
print(f"\n=== HOÀN TẤT LƯỚI ĐÁNH GIÁ N7: Chạy trong {total_elapsed/60:.2f} phút (Bỏ qua {skipped}/{total_configs} file có sẵn) ===")

In [ ]:
## 5. Tổng hợp Kết quả (Mean ± Std) & Xuất Báo cáo CSV / Markdown

from scripts.summarize_results import load_results, compute_summary, format_table, format_markdown, format_latex
from IPython.display import display, Markdown

# 5.1 Đọc toàn bộ file kết quả trong results/
results_df = load_results("results")
print(f"Tổng số file kết quả đọc được: {len(results_df)}")

# 5.2 Tính Mean ± Std theo từng (dataset, variant)
summary_df = compute_summary(results_df, as_percent=True)

# 5.3 In bảng tổng quan định dạng Text
print(format_table(summary_df, title="BẢNG TỔNG HỢP TOÀN BỘ KẾT QUẢ ĐÁNH GIÁ (ACC %)"))

# 5.4 Hiển thị bảng Markdown chi tiết trong Notebook
print("\n=== BẢNG TỔNG HỢP TOÀN BỘ (MARKDOWN) ===")
display(Markdown(format_markdown(summary_df)))

# 5.5 Các bảng chuyên biệt phục vụ Báo cáo cuối kỳ:
# Bảng 1: Bảng kết quả chính các bản A, B, A+B, C, A+C, D, A+D (Mục 5.3)
main_n7_variants = ["A", "B", "A+B", "C", "A+C", "D", "A+D"]
main_n7_summary = summary_df[summary_df["variant"].isin(main_n7_variants)]
if not main_n7_summary.empty:
    print("\n--- BẢNG 1: BẢNG KẾT QUẢ CHÍNH (MỤC 5.3 BÁO CÁO - SO SÁNH A, B, C, D VÀ CÁC BẢN GHÉP) ---")
    display(Markdown(format_markdown(main_n7_summary)))

# Bảng 2: Phân tích độ nhạy alpha in {0.25, 0.5, 0.75} cho A+C và A+D (Mục 5.5)
alpha_n7_variants = [
    "A+C_a0.25", "A+C", "A+C_a0.75",
    "A+D_a0.25", "A+D", "A+D_a0.75",
]
alpha_n7_summary = summary_df[summary_df["variant"].isin(alpha_n7_variants)]
if not alpha_n7_summary.empty:
    print("\n--- BẢNG 2: PHÂN TÍCH ĐỘ NHẠY ALPHA CHO A+C VÀ A+D (MỤC 5.5 BÁO CÁO) ---")
    display(Markdown(format_markdown(alpha_n7_summary)))

# 5.6 Lưu file tổng hợp CSV, Markdown và chi tiết từng seed (cho Phụ lục B)
summary_df.to_csv("results/summary_n7.csv", index=False, encoding="utf-8")
with open("results/summary_n7.md", "w", encoding="utf-8") as f:
    f.write(format_markdown(summary_df))

# Lưu chi tiết từng seed phục vụ Phụ lục B của báo cáo
detailed_df = results_df.copy()
for col in ["all", "old", "new"]:
    detailed_df[f"{col}_pct"] = (detailed_df[col] * 100).round(2)
detailed_df.to_csv("results/detailed_seeds_n7.csv", index=False, encoding="utf-8")

print("\n✔ Đã lưu: results/summary_n7.csv")
print("✔ Đã lưu: results/summary_n7.md")
print("✔ Đã lưu: results/detailed_seeds_n7.csv (Dữ liệu cho Phụ lục B)")

In [ ]:
## 6. Đóng Gói Kết Quả Nộp Bài (/kaggle/working/results_n7.zip)

import shutil

zip_out = shutil.make_archive("/kaggle/working/results_n7", "zip", "results")
zip_size_kb = os.path.getsize(zip_out) / 1024

print("=== ĐÃ ĐÓNG GÓI KẾT QUẢ N7 THÀNH CÔNG ===")
print(f"📦 File nén: {zip_out} ({zip_size_kb:.1f} KB)")
print(f"📂 Số lượng file trong results/: {len(os.listdir('results'))}")
print("\n👉 Bạn có thể tải file 'results_n7.zip' từ thanh Output bên phải của Kaggle về máy.")